In [35]:
import strawberryfields as sf
from strawberryfields.ops import *
import numpy as np
from scipy.special import erfc
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from helper_functions import protocols
from scipy.optimize import curve_fit

In [36]:
from importlib import reload
reload(protocols)

<module 'helper_functions.protocols' from 'c:\\Users\\kpour\\Desktop\\Github\\Quantum-Communications-Internship-IRIDA-\\helper_functions\\protocols.py'>

# Coherent States Noise-Free

## Produce Data

In [ ]:
alpha_grid = np.arange(0, 1, 0.01)
N_cs, p_err_cs = protocols.perr_cs(alpha_grid=alpha_grid, homodyne_angle=0, num_samples=1000)

np.savez("data/perr_data_cs_a100_S1000.npz", N_cs=N_cs, alpha_grid=alpha_grid, p_err_cs=p_err_cs)

## Load Data

In [22]:
data = np.load("data/perr_data_cs_a100_S1000.npz")

N_cs = data["N_cs"]
alpha_grid = data["alpha_grid"]
p_err_cs = data["p_err_cs"]
beta_cs = 1 - N_cs/np.max(N_cs)

## Plot Data + Theory

In [68]:

# Theoretical surface
N_mesh, beta_mesh = np.meshgrid(N_cs, beta_cs, indexing="ij")
p_err_theory = protocols.perr_homodyne(N_mesh, beta=0)

z_theory = np.log10(p_err_theory)
z_sim = np.log10(p_err_cs)

# Figure
fig = go.Figure()

# Theory surface
fig.add_trace(
    go.Surface(x=N_mesh, y=beta_mesh, z=z_theory, colorscale="Greys_r", opacity=0.5, name="Theory", colorbar=dict(title="log₁₀(P_err)"))
)

# Simulation scatter extended in beta
beta_values = np.linspace(0, 1, 20)

for beta in beta_values:
    fig.add_trace(
        go.Scatter3d(x=N_cs, y=np.full_like(N_cs, beta), z=z_sim, mode="markers",
            marker=dict(
                size=3,
                color=z_sim,
                colorscale="Viridis",
                opacity=1
            ),
            name=f"Simulation β={beta:.2f}",
            showlegend=False
        )
    )

# Layout
fig.update_layout(
    title="Error probability",
    width=800,
    height=800,
    scene=dict(
        xaxis_title="N",
        yaxis_title="β",
        zaxis_title="log₁₀(P_err)"
    )
)

fig.show()

## Plot fit of data to theory

In [69]:
# Fit theoretical form
z_data = np.log10(p_err_cs)

def model(N, A, B):
    return np.log10(A * erfc(np.sqrt(B*N)))

params, covariance = curve_fit(model, N_cs, z_data)
A_fit, B_fit = params
A_err, B_err = np.sqrt(np.diag(covariance))

print(f"A = {A_fit:.3f} ± {A_err:.3f}")
print(f"B = {B_fit:.3f} ± {B_err:.3f}")

# Create fitted surface
N_fit = np.linspace(N_cs.min(), N_cs.max(), 200)
beta_fit = np.linspace(0, 1, 200)
N_surface, beta_surface = np.meshgrid(N_fit, beta_fit, indexing="ij")

# No beta dependence
z_surface = model(N_surface, A_fit, B_fit)


# Plot
fig = go.Figure()

# Fitted surface
fig.add_trace(
    go.Surface(x=N_surface, y=beta_surface, z=z_surface, colorscale="Viridis", opacity=1, name="Fitted surface")
)

fig.update_layout(
    title="Fitted error probability surface",
    width=800,
    height=800,
    scene=dict(xaxis_title="N", yaxis_title="β", zaxis_title="log₁₀(P_err)")
)


fig.show()

A = 0.501 ± 0.009
B = 1.994 ± 0.030


# DDS Noise-Free

## Produce Data

In [ ]:
N_grid = np.arange(0, 1, 0.02)
beta_grid = np.arange(0, 1, 0.02)
num_samples=1000
p_err = protocols.perr_dss(N_grid=N_grid, beta_grid=beta_grid, homodyne_angle=0, num_samples=num_samples)
np.savez(f"data/perr_data_dss_N{len(N_grid)}_b{len(beta_grid)}_S{num_samples}.npz", N_grid=N_grid, beta_grid=beta_grid, p_err=p_err)

## Load data

In [55]:
data = np.load("data/perr_data_dss_N50_b50_S2000.npz")

N_grid = data["N_grid"]
beta_grid = data["beta_grid"]
p_err = data["p_err"]

## Plot Data + Theory

In [57]:
# Create Mesh
N_mesh, beta_mesh = np.meshgrid(N_grid, beta_grid, indexing='ij')
p_err_theory = protocols.perr_homodyne(N_mesh, beta_mesh)

# Log scale for the probability
z_sim = np.log(p_err)
z_theory = np.log(p_err_theory)
zmin = np.floor(np.min(z_theory))
zmax = np.ceil(np.max(z_theory))
ticks = np.arange(zmin, zmax + 1)

# Figure
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=N_mesh.ravel(), y=beta_mesh.ravel(), z=z_sim.ravel(), mode='markers', marker=dict(size=3, color=z_sim.ravel(), colorscale='Viridis', opacity=1), name="Simulation"))
fig.add_trace(go.Surface(x=N_mesh, y=beta_mesh, z=z_theory, colorscale="Greys_r", opacity=0.5, colorbar=dict(title="log(P_err)")))

# Figure settings
fig.update_layout(
    title="Error probability", 
    scene=dict(xaxis_title="N", 
               yaxis_title="β",  
               zaxis=dict(title="log(P_err)", 
               tickmode="array", 
                tickvals=ticks, ticktext=[f"10e{int(t)}" for t in ticks])), 
                width=800, 
                height=800
                )
fig.show()

## Plot fit of data to theory

In [ ]:
N_mesh, beta_mesh = np.meshgrid(N_grid, beta_grid, indexing="ij")

# Define fit function
def model(coords, A, B):

    N, beta = coords
    alpha = np.sqrt(N*(1-beta))
    Sigma = 1 / (np.sqrt(N*beta) +np.sqrt(1 + N*beta))

    return (A * erfc(B*alpha/Sigma)).ravel()

# Fit A and B
params, covariance = curve_fit(model,(N_mesh, beta_mesh), p_err.ravel(), p0=[np.sqrt(2), 0.5])
A_fit, B_fit = params
A_err, B_err = np.sqrt(np.diag(covariance))
print(f"A = {A_fit:.3f} ± {A_err:.3f}")
print(f"B = {B_fit:.3f} ± {B_err:.3f}")


# Generate fitted surface
N_fit = np.linspace(N_mesh.min(), N_mesh.max(), 200)
beta_fit = np.linspace(beta_mesh.min(), beta_mesh.max(), 200)
N_surface, beta_surface = np.meshgrid(N_fit, beta_fit, indexing="ij")
p_surface = model((N_surface, beta_surface), A_fit, B_fit).reshape(N_surface.shape)

# Log scale
z_sim = np.log(p_err.ravel())
z_surface = np.log(p_surface)

# Use surface + data range
zmin = np.floor(min(z_sim.min(), z_surface.min()))
zmax = np.ceil(max(z_sim.max(), z_surface.max()))
ticks = np.arange(zmin, zmax+1)


# Plot
fig = go.Figure()


# Fitted surface
fig.add_trace(
    go.Surface(x=N_surface, y=beta_surface, z=z_surface, colorscale="Inferno", opacity=1, name="Fit")
)

# Layout
fig.update_layout(
    title="Error probability fit",
    width=900,
    height=800,
    scene=dict(
        xaxis_title="N",
        yaxis_title="β",
        zaxis=dict(
            title="log(P_err)",
            tickmode="array",
            tickvals=ticks,
            ticktext=[f"10e{int(t)}" for t in ticks]
        )
    )
)


fig.show()

A = 0.500 ± 0.001
B = 1.415 ± 0.002
